# Section 3: Low NA Annular Aperture

**System:** NA = 0.1, λ = 532 nm, annular aperture, medium: air (n = 1).

An annular aperture blocks the central disk (obscuration ratio ε) and transmits only the outer ring. This trades peak intensity for a narrower central lobe and extended depth of field — a classic trade-off in optical design.

**Goals:**
- Focal-plane 1D radial intensity for ε = 0.5 and ε = 0.99, with uniform and Gaussian inputs
- Axial intensity I(r=0, z) — depth of field effect
- Compare annular vs full-disk (ε = 0) profiles
- Use `annular_intensity()` from the package

**Method:** `annular_intensity()` applies Babinet's principle:
$$E_{annular} = E_{outer}(NA) - E_{inner}(\varepsilon \cdot NA)$$

> **Note:** Each `annular_intensity()` call computes two RW integrals (outer + inner disk). Use small arrays (50–70 points) for reasonable speed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/Users/raaromero/Projects/Research/optical-diffraction/src')

from optical_diffraction import RichardsWolfSimulator, annular_intensity

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})

# System parameters
NA = 0.1
wavelength = 0.532   # microns
n_medium = 1.0
airy_radius = 0.61 * wavelength / NA
dof = wavelength / NA**2

print(f"System: NA={NA}, λ={wavelength} μm")
print(f"Airy radius: {airy_radius:.3f} μm")
print(f"Depth of focus (λ/NA²): {dof:.2f} μm")

## 3.1 Focal Plane Intensity — ε = 0.5 (Moderate Obscuration)

Comparing uniform and Gaussian inputs through an annular aperture with ε = 0.5.

In [ ]:
# Radial grid at focal plane
r_max = 5.0 * airy_radius
r = np.linspace(0, r_max, 70)
z_focal = np.zeros_like(r)

epsilon_05 = 0.5

# Input field configurations
configs = [
    ('uniform', 0.0, 'Uniform', 'tab:blue', '-'),
    ('gaussian', 1.0, 'Gaussian α=1', 'tab:orange', '-'),
    ('gaussian', 2.0, 'Gaussian α=2', 'tab:green', '-'),
    ('gaussian', 4.0, 'Gaussian α=4', 'tab:red', '-'),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Full disk (ε=0) for comparison
ax = axes[0]
ax.set_title(f'Full Disk (ε=0) — Reference')
ax2 = axes[1]
ax2.set_title(f'Annular (ε={epsilon_05})')

for (field_type, trunc, label, color, ls) in configs:
    # Full disk (ε=0)
    I_disk = annular_intensity(
        NA=NA, epsilon=0.0, r=r, z=z_focal,
        wavelength=wavelength, n_medium=n_medium,
        input_field=field_type, truncation_coeff=trunc,
    )
    axes[0].plot(r / airy_radius, I_disk, color=color, lw=2, label=label)

    # Annular ε=0.5
    I_ann = annular_intensity(
        NA=NA, epsilon=epsilon_05, r=r, z=z_focal,
        wavelength=wavelength, n_medium=n_medium,
        input_field=field_type, truncation_coeff=trunc,
    )
    axes[1].plot(r / airy_radius, I_ann, color=color, lw=2, label=label)

for ax in axes:
    ax.axvline(x=1.0, color='black', ls=':', lw=1.2, alpha=0.5, label='Airy radius')
    ax.set_xlabel('r / r$_{Airy}$')
    ax.set_ylabel('Normalized intensity')
    ax.set_xlim(0, r_max / airy_radius)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

fig.suptitle(f'Focal Plane Radial Intensity — Low NA (NA={NA}), Annular Aperture ε={epsilon_05}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3.2 Focal Plane Intensity — ε = 0.99 (Strong Annular)

At ε = 0.99 the aperture is nearly a thin ring. This produces a very narrow central lobe (approaching the "super-resolution" limit) but at the cost of strong side lobes.

In [ ]:
epsilon_099 = 0.99

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for (field_type, trunc, label, color, ls) in configs:
    # Full disk reference
    I_disk = annular_intensity(
        NA=NA, epsilon=0.0, r=r, z=z_focal,
        wavelength=wavelength, n_medium=n_medium,
        input_field=field_type, truncation_coeff=trunc,
    )
    axes[0].plot(r / airy_radius, I_disk, color=color, lw=2, label=label)

    # Annular ε=0.99
    I_ann = annular_intensity(
        NA=NA, epsilon=epsilon_099, r=r, z=z_focal,
        wavelength=wavelength, n_medium=n_medium,
        input_field=field_type, truncation_coeff=trunc,
    )
    axes[1].plot(r / airy_radius, I_ann, color=color, lw=2, label=label)

axes[0].set_title('Full Disk (ε=0) — Reference')
axes[1].set_title(f'Annular (ε={epsilon_099})')

for ax in axes:
    ax.axvline(x=1.0, color='black', ls=':', lw=1.2, alpha=0.5, label='Airy radius')
    ax.set_xlabel('r / r$_{Airy}$')
    ax.set_ylabel('Normalized intensity')
    ax.set_xlim(0, r_max / airy_radius)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

fig.suptitle(f'Focal Plane Radial Intensity — Low NA (NA={NA}), Annular Aperture ε={epsilon_099}', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 3.3 Axial Intensity — Depth of Field Effect

Annular apertures are well known to extend the depth of field. We compare the on-axis profiles for ε = 0, 0.5, 0.99 using uniform illumination.

In [ ]:
# Axial grid
z_max = 4.0 * dof
z_axial = np.linspace(-z_max, z_max, 60)
r_zero = np.zeros_like(z_axial)

epsilons = [0.0, 0.5, 0.99]
eps_colors = ['tab:blue', 'tab:orange', 'tab:red']
eps_styles = ['-', '--', ':']
eps_labels = ['ε=0 (full disk)', 'ε=0.5', 'ε=0.99']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Uniform input
ax = axes[0]
ax.set_title('Axial Intensity — Uniform Input')
for eps, color, ls, label in zip(epsilons, eps_colors, eps_styles, eps_labels):
    I_ax = annular_intensity(
        NA=NA, epsilon=eps, r=r_zero, z=z_axial,
        wavelength=wavelength, n_medium=n_medium,
        input_field='uniform', truncation_coeff=0.0,
    )
    ax.plot(z_axial / dof, I_ax, color=color, lw=2.5, ls=ls, label=label)

# Gaussian α=2 input
ax = axes[1]
ax.set_title('Axial Intensity — Gaussian Input (α=2)')
for eps, color, ls, label in zip(epsilons, eps_colors, eps_styles, eps_labels):
    I_ax = annular_intensity(
        NA=NA, epsilon=eps, r=r_zero, z=z_axial,
        wavelength=wavelength, n_medium=n_medium,
        input_field='gaussian', truncation_coeff=2.0,
    )
    ax.plot(z_axial / dof, I_ax, color=color, lw=2.5, ls=ls, label=label)

for ax in axes:
    ax.axvline(x=0, color='black', ls='-', lw=0.8, alpha=0.4)
    ax.axhline(y=0.5, color='gray', ls=':', lw=1, alpha=0.6, label='50% level')
    ax.set_xlabel('z / DoF  (DoF = λ/NA²)')
    ax.set_ylabel('Normalized on-axis intensity I(r=0,z)')
    ax.legend(fontsize=9)

fig.suptitle(f'Axial Intensity — Depth of Field Effect (NA={NA})', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print("Key result: Larger ε → longer depth of field, but reduced peak intensity.")

## 3.4 Direct Comparison: Full Disk vs Annular (Overlay)

Overlay of focal-plane profiles for ε = 0 vs ε = 0.5 vs ε = 0.99, for each input field type.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

field_cases = [
    ('uniform', 0.0, 'Uniform'),
    ('gaussian', 1.0, 'Gaussian α=1'),
    ('gaussian', 2.0, 'Gaussian α=2'),
    ('gaussian', 4.0, 'Gaussian α=4'),
]

for ax, (field_type, trunc, field_label) in zip(axes, field_cases):
    for eps, color, ls, eps_label in zip(epsilons, eps_colors, eps_styles, eps_labels):
        I = annular_intensity(
            NA=NA, epsilon=eps, r=r, z=z_focal,
            wavelength=wavelength, n_medium=n_medium,
            input_field=field_type, truncation_coeff=trunc,
        )
        ax.plot(r / airy_radius, I, color=color, lw=2, ls=ls, label=eps_label)

    ax.axvline(x=1.0, color='black', ls=':', lw=1.0, alpha=0.5)
    ax.set_xlabel('r / r$_{Airy}$')
    ax.set_ylabel('Normalized intensity')
    ax.set_title(f'Input: {field_label}')
    ax.set_xlim(0, r_max / airy_radius)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

fig.suptitle(f'Focal Plane: Full Disk vs Annular Apertures — Low NA (NA={NA})', fontsize=13)
plt.tight_layout()
plt.show()

## 3.5 All Gaussian Inputs — Annular ε = 0.5 and ε = 0.99 Side by Side

In [ ]:
alphas_plot = [1, 2, 4]
colors_alpha = ['tab:orange', 'tab:green', 'tab:red']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for eps_val, ax in zip([0.5, 0.99], axes):
    # Include uniform as baseline
    I_unif = annular_intensity(
        NA=NA, epsilon=eps_val, r=r, z=z_focal,
        wavelength=wavelength, n_medium=n_medium,
        input_field='uniform', truncation_coeff=0.0,
    )
    ax.plot(r / airy_radius, I_unif, color='tab:blue', lw=2, label='Uniform')

    for alpha, color in zip(alphas_plot, colors_alpha):
        I = annular_intensity(
            NA=NA, epsilon=eps_val, r=r, z=z_focal,
            wavelength=wavelength, n_medium=n_medium,
            input_field='gaussian', truncation_coeff=float(alpha),
        )
        ax.plot(r / airy_radius, I, color=color, lw=2, label=f'Gaussian α={alpha}')

    ax.axvline(x=1.0, color='black', ls=':', lw=1.0, alpha=0.5)
    ax.set_xlabel('r / r$_{Airy}$')
    ax.set_ylabel('Normalized intensity')
    ax.set_title(f'Annular Aperture ε={eps_val} — All Inputs')
    ax.set_xlim(0, r_max / airy_radius)
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

fig.suptitle(f'Focal Plane Intensity: Gaussian Inputs Through Annular Apertures (NA={NA})', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Summary

| Effect | ε=0 (full disk) | ε=0.5 | ε=0.99 |
|---|---|---|---|
| Central lobe width | Widest | Narrower | Narrowest |
| Side lobe level | Lowest (Airy) | Higher | Highest |
| Peak intensity | Highest | Lower | Much lower |
| Depth of field | Shortest | Extended | Most extended |

At low NA, annular apertures provide the trade-off familiar from scalar diffraction theory: narrower central focus at the cost of reduced peak intensity and stronger side lobes. The depth of field is extended because the annular geometry selects a restricted range of convergence angles.

The next notebook (04) repeats this analysis at high NA (NA=0.9) where vectorial effects become significant.